# AI_LUNG — Full Training Pipeline (Google Colab)

Runs all three modules end-to-end on the full LIDC-IDRI dataset:
1. 2.5D Denoising
2. 3D Reconstruction
3. Nodule Detection

**Before running:**
- Upload to Google Drive at: `MyDrive/AI_LUNG_DATA/`
  - `MyDrive/AI_LUNG_DATA/manifest.zip` (67 GiB — DICOM scans)
  - `MyDrive/AI_LUNG_DATA/LIDC-XML-only.zip` (small — annotations)
- Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)

> **First run only:** Step 1c will unzip both archives to Drive (~20-30 min one-time cost).

## Step 1: Check GPU & Mount Drive

In [ ]:
import torch

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# Step 1a: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# Step 1b: Unzip dataset archives (first run only — ~20-30 min each; skipped if already extracted)
import os, subprocess

data_root = '/content/drive/MyDrive/AI_LUNG_DATA'

# --- Unzip manifest (DICOM scans, 67 GiB) ---
manifest_extracted = f'{data_root}/manifest-1600709154662'
manifest_zip       = f'{data_root}/manifest.zip'

print('Extracting manifest.zip → Drive (skips already-extracted files, safe to resume)...')
result = subprocess.run(
    ['unzip', '-n', '-q', manifest_zip, '-d', data_root],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('UNZIP ERROR:', result.stderr)
else:
    print('manifest.zip done (new files extracted, existing files skipped).')

# --- Unzip XML annotations (~50 MB) ---
xml_extracted = f'{data_root}/LIDC-XML-only'
xml_zip       = f'{data_root}/LIDC-XML-only.zip'

print('Extracting LIDC-XML-only.zip...')
result = subprocess.run(
    ['unzip', '-n', '-q', xml_zip, '-d', data_root],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('UNZIP ERROR:', result.stderr)
else:
    print('LIDC-XML-only.zip done.')

In [ ]:
# Step 1c: Verify dataset layout
import os

data_root = '/content/drive/MyDrive/AI_LUNG_DATA'
assert os.path.exists(data_root), f'Dataset folder not found: {data_root}'
assert os.path.exists(f'{data_root}/manifest-1600709154662/metadata.csv'), \
    'metadata.csv missing — did unzip finish?'
assert os.path.exists(f'{data_root}/LIDC-XML-only'), \
    'LIDC-XML-only folder missing — did unzip finish?'

# Count patients as sanity check
lidcdir = f'{data_root}/manifest-1600709154662/LIDC-IDRI'
patients = [d for d in os.listdir(lidcdir) if d.startswith('LIDC-IDRI-')]
print(f'Dataset verified. Found {len(patients)} patient folders.')

## Step 2: Clone Repository & Install Dependencies

In [ ]:
import os

repo_dir = '/content/AI_LUNG'

if os.path.exists(repo_dir):
    # Pull latest changes if already cloned
    %cd /content/AI_LUNG
    !git pull origin main
else:
    !git clone https://github.com/KailasVS666/AI_LUNG.git /content/AI_LUNG
    %cd /content/AI_LUNG

print('Repository ready.')

In [ ]:
%cd /content/AI_LUNG
!pip install -q -r requirements.txt
!pip install -q -e .
print('Dependencies installed.')

## Step 3: Generate Patient-wise Data Splits

Splits patients 70/15/15 into train/val/test with no leakage.
Skipped if split file already exists.

In [ ]:
import os, json

split_path = '/content/AI_LUNG/outputs/splits/patient_split.json'

if os.path.exists(split_path):
    with open(split_path) as f:
        splits = json.load(f)
    print(f'Splits already exist — train: {len(splits["train"])}, val: {len(splits["val"])}, test: {len(splits["test"])}')
else:
    !python scripts/build_splits.py \
        --dataset-root /content/drive/MyDrive/AI_LUNG_DATA/manifest-1600709154662 \
        --metadata-csv /content/drive/MyDrive/AI_LUNG_DATA/manifest-1600709154662/metadata.csv \
        --out outputs/splits/patient_split.json \
        --train-ratio 0.70 \
        --val-ratio 0.15
    print('Splits generated.')

In [ ]:
# [OPTIONAL] If you already have a split JSON from your local Windows machine,
# run this cell to remap the hardcoded Windows paths to Colab Drive paths.
# Skip if you just ran build_splits.py above (paths are already correct).

import json, re

split_path = '/content/AI_LUNG/outputs/splits/patient_split.json'
LOCAL_PREFIX  = r'[A-Za-z]:[/\\].*?manifest-1600709154662'
COLAB_PREFIX  = '/content/drive/MyDrive/AI_LUNG_DATA/manifest-1600709154662'

with open(split_path) as f:
    splits = json.load(f)

def remap(entry):
    loc = entry.get('file_location', '')
    entry['file_location'] = re.sub(LOCAL_PREFIX, COLAB_PREFIX, str(loc).replace('\\', '/'))
    return entry

for key in ('train', 'val', 'test'):
    splits[key] = [remap(e) for e in splits[key]]

with open(split_path, 'w') as f:
    json.dump(splits, f, indent=2)

print(f'Paths remapped. Sample: {splits[\"train\"][0][\"file_location\"]}')

## Step 4: Train 2.5D Denoiser

**Model**: 2-level 2D U-Net, 3-slice input → denoised center slice  
**Metrics**: PSNR, SSIM  
**Config**: `configs/baseline_colab.yaml`  
**Estimated time**: ~2-4 hours (T4 GPU, full dataset, 30 epochs)

In [ ]:
%cd /content/AI_LUNG
!python scripts/train_denoiser_baseline.py --config configs/baseline_colab.yaml

In [ ]:
# Show denoising training results
import json
import matplotlib.pyplot as plt

history_path = '/content/drive/MyDrive/AI_LUNG_DATA/outputs/baseline_25d/history.json'
with open(history_path) as f:
    h = json.load(f)

epochs = range(1, len(h['val_psnr']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, h['train_loss'], label='Train'); axes[0].plot(epochs, h['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, h['val_psnr'], 'g-o')
axes[1].set_title(f'Val PSNR (best: {max(h["val_psnr"]):.2f} dB)'); axes[1].grid(True)

axes[2].plot(epochs, h['val_ssim'], 'b-o')
axes[2].set_title(f'Val SSIM (best: {max(h["val_ssim"]):.4f})'); axes[2].grid(True)

plt.tight_layout(); plt.show()
print(f'Best PSNR: {max(h["val_psnr"]):.2f} dB | Best SSIM: {max(h["val_ssim"]):.4f}')

## Step 5: Train 3D Reconstruction Model

**Model**: 3D Attention U-Net with physics-guided loss  
**Metrics**: PSNR, SSIM  
**Config**: `configs/recon3d_colab.yaml`  
**Estimated time**: ~4-6 hours (T4 GPU, full dataset, 30 epochs)

In [ ]:
%cd /content/AI_LUNG
!python scripts/train_recon3d.py --config configs/recon3d_colab.yaml

In [ ]:
# Show 3D reconstruction results
import json
import matplotlib.pyplot as plt

history_path = '/content/drive/MyDrive/AI_LUNG_DATA/outputs/recon3d/history.json'
with open(history_path) as f:
    h = json.load(f)

epochs = range(1, len(h['val_psnr']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, h['train_loss'], label='Train'); axes[0].plot(epochs, h['val_loss'], label='Val')
axes[0].set_title('Total Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, h['val_psnr'], 'g-o')
axes[1].set_title(f'Val PSNR (best: {max(h["val_psnr"]):.2f} dB)'); axes[1].grid(True)

axes[2].plot(epochs, h['val_ssim'], 'b-o')
axes[2].set_title(f'Val SSIM (best: {max(h["val_ssim"]):.4f})'); axes[2].grid(True)

plt.tight_layout(); plt.show()
print(f'Best PSNR: {max(h["val_psnr"]):.2f} dB | Best SSIM: {max(h["val_ssim"]):.4f}')

## Step 6: Train Nodule Detector

**Model**: 4-level 3D CNN with global average pooling  
**Metrics**: ROC AUC, Sensitivity, Specificity  
**Config**: `configs/nodule_detection_colab.yaml`  
**Estimated time**: ~3-5 hours (T4 GPU, full dataset, 30 epochs)

In [ ]:
%cd /content/AI_LUNG
!python scripts/train_nodule_detector.py --config configs/nodule_detection_colab.yaml

In [ ]:
# Show detection results
import json
import matplotlib.pyplot as plt

history_path = '/content/drive/MyDrive/AI_LUNG_DATA/outputs/nodule_detection/history.json'
with open(history_path) as f:
    h = json.load(f)

epochs = range(1, len(h['train_loss']) + 1)
aucs  = [m['auc']         for m in h['val_metrics']]
sens  = [m['sensitivity'] for m in h['val_metrics']]
spec  = [m['specificity'] for m in h['val_metrics']]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, h['train_loss'], 'r-o')
axes[0].set_title('Train Loss'); axes[0].grid(True)

axes[1].plot(epochs, aucs,  label=f'AUC (best: {max(aucs):.4f})',  marker='o')
axes[1].plot(epochs, sens,  label=f'Sensitivity (best: {max(sens):.4f})', marker='s')
axes[1].plot(epochs, spec,  label=f'Specificity (best: {max(spec):.4f})', marker='^')
axes[1].set_title('Validation Metrics'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout(); plt.show()
print(f'Best AUC: {max(aucs):.4f} | Best Sensitivity: {max(sens):.4f} | Best Specificity: {max(spec):.4f}')

## Step 7: Final Results Summary

In [ ]:
import json

base_dir = '/content/drive/MyDrive/AI_LUNG_DATA/outputs'

# Load all histories
with open(f'{base_dir}/baseline_25d/history.json')    as f: h_den  = json.load(f)
with open(f'{base_dir}/recon3d/history.json')          as f: h_rec  = json.load(f)
with open(f'{base_dir}/nodule_detection/history.json') as f: h_det  = json.load(f)

aucs = [m['auc'] for m in h_det['val_metrics']]
sens = [m['sensitivity'] for m in h_det['val_metrics']]
spec = [m['specificity'] for m in h_det['val_metrics']]

print('=' * 55)
print('         AI_LUNG — FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  2.5D Denoising   | PSNR: {max(h_den["val_psnr"]):.2f} dB  | SSIM: {max(h_den["val_ssim"]):.4f}')
print(f'  3D Reconstruction| PSNR: {max(h_rec["val_psnr"]):.2f} dB  | SSIM: {max(h_rec["val_ssim"]):.4f}')
print(f'  Nodule Detection | AUC:  {max(aucs):.4f}     | Sens: {max(sens):.4f} | Spec: {max(spec):.4f}')
print('=' * 55)

## Step 8: Save Checkpoints to Drive

All checkpoints are already saved directly to Drive via `output_dir` in configs.  
Run this cell to verify all are present.

In [ ]:
import os

base = '/content/drive/MyDrive/AI_LUNG_DATA/outputs'
checkpoints = [
    f'{base}/baseline_25d/denoiser_best.pt',
    f'{base}/baseline_25d/denoiser_last.pt',
    f'{base}/recon3d/recon3d_best.pt',
    f'{base}/recon3d/recon3d_last.pt',
    f'{base}/nodule_detection/nodule_detector_best.pt',
    f'{base}/nodule_detection/nodule_detector_last.pt',
]

all_ok = True
for cp in checkpoints:
    exists = os.path.exists(cp)
    size_mb = os.path.getsize(cp) / 1e6 if exists else 0
    status = f'{size_mb:.1f} MB' if exists else 'MISSING'
    print(f'  {"OK" if exists else "!!"}  {os.path.basename(cp):40s} {status}')
    if not exists:
        all_ok = False

print()
print('All checkpoints saved to Drive.' if all_ok else 'Some checkpoints are missing — re-run training cells.')